In [13]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [14]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a              
elm_

In [15]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

#

[설정] 경로가 수정된 임시 XML 생성: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [16]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [17]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


실험

In [18]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver

# 1. 설정 및 초기화
target_pos_412 = np.array([418.28, 1.50, -308.20])
print("="*60 + f"\n[설정] 테스트 목표: {target_pos_412}\n" + "="*60)

# 기존 객체 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3', 'Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

# 2. 장치 배치
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(target_pos_412)
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료.")

rx = Receiver(name="rx_car", position=target_pos_412)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 배치 완료.")

# 3. 시뮬레이션
print("[연산] 경로 계산 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 4. 결과 검증
a, tau = paths.cir()
if tf.size(a) > 0 and tf.reduce_sum(tf.abs(a)) > 0:
    print(f"\n✅ [성공] 전파 도달 (Amp Sum: {tf.reduce_sum(tf.abs(a)):.2e})")
else:
    print("\n❌ [실패] 전파 미도달 (장애물 또는 거리 문제)")

# 5. 시각화
cam = Camera(position=target_pos_412 + np.array([0, 100, 100]), look_at=target_pos_412)
print("[시각화] 3D 뷰어 실행")
scene.preview(paths=paths, show_devices=True)

[설정] 테스트 목표: [ 418.28    1.5  -308.2 ]
 -> Tx_1 배치 완료.
 -> Tx_2 배치 완료.
 -> Tx_3 배치 완료.
 -> 자동차(Rx) 배치 완료.
[연산] 경로 계산 시작...

❌ [실패] 전파 미도달 (장애물 또는 거리 문제)
[시각화] 3D 뷰어 실행


In [19]:
# ==============================================================================
# 3. 경로(Trajectory) 좌표 정밀 보정 (Raw Data Inspection)
# ==============================================================================
print("[보정] 빨간색 도로 좌표 정밀 분석 시작...")

road_positions = []
target_road_id = "elm__00"

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                # 1) 원본 좌표 추출
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # 2) [진단] 412번 인덱스의 '원본(Raw)' 좌표 확인
                if len(v_pos) > 412:
                    raw_412 = v_pos[412]
                    print(f" -> [진단] Raw Index 412: {raw_412}")
                    # 예상: [418.xx, 0.0, -308.xx] 또는 [418.xx, -308.xx, 0.0] 등
                    
                    # 3) [해결] 목표 좌표(Target)와 비교하여 매핑 결정
                    # Target Z: -308.20
                    
                    # Case A: Raw Y가 -308 근처인 경우 -> Y를 Z로 (x, 0, y)
                    if np.isclose(raw_412[1], -308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 Z축으로 이동합니다. (Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]
                        
                    # Case B: Raw Z가 -308 근처인 경우 -> 그대로 사용 (Z -> Z)
                    elif np.isclose(raw_412[2], -308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 그대로 사용합니다. (No Rotation)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 2]
                        
                    # Case C: Raw Y가 308 근처인 경우 -> 부호 반전 후 이동 (-Y -> Z)
                    elif np.isclose(raw_412[1], 308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 반전하여 Z축으로 이동합니다. (-Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 1]

                    # Case D: Raw Z가 308 근처인 경우 -> 부호 반전 (-Z -> Z)
                    elif np.isclose(raw_412[2], 308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 반전하여 사용합니다. (-Z -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 2]
                        
                    else:
                        print(" -> [경고] 자동 매핑 실패. Raw 데이터가 예상과 다릅니다. 원본 그대로 사용합니다.")
                        # 기본: (x, 0, y) 시도 (가장 흔한 패턴)
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]

                print(f" -> 좌표 변환 완료 ({len(road_positions)} vertices)")
            break

[보정] 빨간색 도로 좌표 정밀 분석 시작...
 -> [진단] Raw Index 412: [ 4.1827621e+02  1.8871996e-14 -3.0820306e+02]
 -> [결정] Z축 데이터를 그대로 사용합니다. (No Rotation)
 -> 좌표 변환 완료 (610 vertices)


412INDEX 에서 Path Loss

In [20]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver

# 1. 설정 및 초기화
target_pos_522 = road_positions[412]
print("="*60 + f"\n[설정] 테스트 목표: {target_pos_522}\n" + "="*60)

# 기존 객체 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3', 'Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

# 2. 장치 배치
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(target_pos_522)
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료.")

rx = Receiver(name="rx_car", position=target_pos_522)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 배치 완료.")

# 3. 시뮬레이션
print("[연산] 경로 계산 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 4. 결과 검증
a, tau = paths.cir()
if tf.size(a) > 0 and tf.reduce_sum(tf.abs(a)) > 0:
    print(f"\n✅ [성공] 전파 도달 (Amp Sum: {tf.reduce_sum(tf.abs(a)):.2e})")
else:
    print("\n❌ [실패] 전파 미도달 (장애물 또는 거리 문제)")

# 5. 시각화
cam = Camera(position=target_pos_522 + np.array([0, 100, 100]), look_at=target_pos_522)
print("[시각화] 3D 뷰어 실행")
scene.preview(paths=paths, show_devices=True)

[설정] 테스트 목표: [ 418.2762     0.      -308.20306]
 -> Tx_1 배치 완료.
 -> Tx_2 배치 완료.
 -> Tx_3 배치 완료.
 -> 자동차(Rx) 배치 완료.
[연산] 경로 계산 시작...

❌ [실패] 전파 미도달 (장애물 또는 거리 문제)
[시각화] 3D 뷰어 실행


차량이 움직이는 경우

In [21]:
import numpy as np
import mitsuba as mi
from sionna.rt import Transmitter, PlanarArray, load_scene, Camera
import os
import matplotlib.pyplot as plt

# ==============================================================================
# 1. 초기 설정 및 데이터 준비
# ==============================================================================
output_dir = "car_move"
os.makedirs(output_dir, exist_ok=True)
print(f"[설정] 이미지는 '{output_dir}' 폴더에 저장됩니다.")

if 'road_positions' not in globals() or len(road_positions) == 0:
    raise ValueError("road_positions 데이터가 없습니다. 먼저 도로 좌표를 추출해주세요.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

waypoints = road_positions[path_indices].copy() 
waypoints[:, 1] += 1.5  

# 누적 거리 계산
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]

speed_kmh = 60.0
speed_ms = speed_kmh * 1000.0 / 3600.0
total_time = total_distance / speed_ms
delta_t = 0.5

print(f"[시뮬레이션 정보]")
print(f" - 총 거리: {total_distance:.2f} m")
print(f" - 속도: {speed_kmh} km/h")
print(f" - 총 시간: {total_time:.2f} 초")
print(f" - 생성될 프레임 수: {int(total_time / delta_t) + 1} 장")

# ==============================================================================
# 2. 이동 함수
# ==============================================================================
def get_state_at_time(t):
    target_dist = speed_ms * t
    target_dist = np.clip(target_dist, 0, total_distance)
    
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    
    seg_start = cumulative_dists[idx]
    seg_len = segment_dists[idx]
    
    if seg_len == 0:
        return waypoints[idx], np.zeros(3)
    
    ratio = (target_dist - seg_start) / seg_len
    pos = waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])
    
    direction = (waypoints[idx+1] - waypoints[idx]) / seg_len
    vel = direction * speed_ms
    
    return pos, vel

# ==============================================================================
# 3. 씬 객체 배치
# ==============================================================================
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()):
        scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()):
        scene.remove(name)

start_pos, start_vel = get_state_at_time(0.0)
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

tx_car = Transmitter(name="Moving_Car", 
                     position=start_pos.tolist(),
                     orientation=[0,0,0], 
                     color=[0.0, 0.3, 1.0]) 
tx_car.transmit_antenna = scene.tx_array
scene.add(tx_car)

# 카메라 설정
cam_pos = [0, 200, -1000]
cam = Camera(position=cam_pos, look_at=[0, 0, 0])

# ==============================================================================
# 4. 시뮬레이션 루프 (오류 해결형)
# ==============================================================================
current_time = 0.0
frame_idx = 0

print("\n[렌더링 시작] 0.5초 단위로 이미지를 저장합니다...")

# 대화형 모드 끄기 (메모리 절약)
plt.ioff()

while current_time <= total_time + delta_t: 
    # 1. 위치 업데이트
    pos, vel = get_state_at_time(current_time)
    tx_car.position = pos 
    tx_car.velocity = vel
    
    # 2. 렌더링 및 저장
    try:
        # Sionna 렌더링 결과 받기
        render_result = scene.render(camera=cam, num_samples=64, resolution=[640, 480])
        
        filename = os.path.join(output_dir, f"frame_{frame_idx:04d}.png")

        # [핵심 수정] 반환된 결과가 'Figure'인지 '데이터'인지 확인하여 처리
        if hasattr(render_result, 'savefig'): 
            # Case A: 결과가 이미 그림(Figure)인 경우 (현재 오류의 원인 해결)
            render_result.savefig(filename, bbox_inches='tight', pad_inches=0.1)
            plt.close(render_result) # 반드시 닫아줘야 메모리 누수 방지
            
        else:
            # Case B: 결과가 데이터(Tensor)인 경우 (표준 방식)
            img_numpy = np.array(render_result)
            img_numpy = np.clip(img_numpy, 0.0, 1.0) # 밝기 클리핑
            
            fig = plt.figure(figsize=(8, 6))
            plt.imshow(img_numpy)
            plt.axis('off') 
            plt.title(f"Time: {current_time:.1f}s | Speed: {speed_kmh}km/h")
            plt.savefig(filename, bbox_inches='tight', pad_inches=0.1)
            plt.close(fig)

        if frame_idx % 10 == 0:
            print(f" -> Saved {filename} (Location: {pos[0]:.1f}, {pos[1]:.1f}, {pos[2]:.1f})")
            
    except Exception as e:
        print(f" [Error] Frame {frame_idx} 실패: {e}")
        plt.close('all') # 에러 발생 시 열린 창 모두 닫기
    
    current_time += delta_t
    frame_idx += 1

print("\n[완료] 모든 이미지가 'car_move' 폴더에 저장되었습니다.")

FileNotFoundError: [Errno 2] No such file or directory: 'car_move'

In [ ]:
import numpy as np

# ==============================================================================
# 0. 데이터 확인
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다. 먼저 도로 데이터를 로드해주세요.")

# ==============================================================================
# 1. 애니메이션에 사용했던 경로 인덱스
# ==============================================================================
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

# 좌표 추출
waypoints = road_positions[path_indices].copy()
# (옵션) 애니메이션 때 적용했던 높이 보정 (+3.0m)을 적용하고 싶으면 아래 주석 해제
# waypoints[:, 1] += 3.0 

# ==============================================================================
# 2. 좌표 및 거리 계산 출력
# ==============================================================================
print(f"{'No.':<5} | {'Index':<6} | {'X (m)':<10} | {'Y (m)':<10} | {'Z (m)':<10} | {'Dist to Next':<15}")
print("-" * 70)

total_dist = 0.0

for i in range(len(waypoints)):
    pos = waypoints[i]
    
    # 다음 지점까지의 거리 계산 (마지막 점 제외)
    dist = 0.0
    if i < len(waypoints) - 1:
        dist = np.linalg.norm(waypoints[i+1] - waypoints[i])
        total_dist += dist
        dist_str = f"{dist:.2f} m"
    else:
        dist_str = "End"

    # 출력 포맷
    print(f"{i:<5} | {path_indices[i]:<6} | {pos[0]:<10.3f} | {pos[1]:<10.3f} | {pos[2]:<10.3f} | {dist_str}")

print("-" * 70)
print(f"📌 총 이동 거리: {total_dist:.2f} m")
print(f"📌 총 경로 포인트 개수: {len(waypoints)} 개")

# ==============================================================================
# 3. (옵션) CSV 파일로 저장하고 싶다면?
# ==============================================================================
# import pandas as pd
# df = pd.DataFrame(waypoints, columns=['X', 'Y', 'Z'])
# df['Original_Index'] = path_indices
# df.to_csv('car_trajectory.csv', index=False)
# print("\n💾 'car_trajectory.csv' 파일로 저장되었습니다.")

No.   | Index  | X (m)      | Y (m)      | Z (m)      | Dist to Next   
----------------------------------------------------------------------
0     | 412    | 418.276    | 0.000      | -308.203   | 25.20 m
1     | 410    | 399.959    | 0.000      | -290.900   | 24.53 m
2     | 408    | 381.030    | 0.000      | -275.293   | 21.90 m
3     | 406    | 362.083    | 0.000      | -264.309   | 20.34 m
4     | 404    | 342.971    | 0.000      | -257.349   | 20.47 m
5     | 401    | 322.825    | 0.000      | -253.742   | 3.06 m
6     | 400    | 319.801    | 0.000      | -253.267   | 3.03 m
7     | 72     | 316.810    | 0.000      | -252.807   | 11.35 m
8     | 69     | 305.587    | 0.000      | -251.117   | 3.96 m
9     | 68     | 301.660    | 0.000      | -250.610   | 0.11 m
10    | 342    | 301.553    | 0.000      | -250.598   | 1.59 m
11    | 340    | 299.967    | 0.000      | -250.431   | 7.26 m
12    | 338    | 292.754    | 0.000      | -249.652   | 6.26 m
13    | 335    | 286.534    | 0.

??

In [ ]:
import numpy as np
import tensorflow as tf
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver

# ==============================================================================
# 0. 사전 데이터 점검
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 변수가 없습니다. 도로 좌표 추출 코드를 먼저 실행해주세요.")

# ==============================================================================
# 1. 이동 경로(Trajectory) 데이터 준비
# ==============================================================================
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

# 좌표 추출 및 전처리 (사용자 요청대로 단순 추출)
waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5 # 높이 1.5m 띄우기

# ------------------------------------------------------------------------------
# [추가] 60km/h 속도 벡터 계산 (타임라인 활성화의 핵심)
# ------------------------------------------------------------------------------
target_speed_kmh = 60.0
target_speed_ms = target_speed_kmh / 3.6  # km/h -> m/s 변환 (약 16.67 m/s)

# 속도 벡터 배열 초기화
velocities = np.zeros_like(waypoints)

# 각 지점 사이의 방향을 계산하여 속도 부여
for i in range(len(waypoints) - 1):
    # 현재 점과 다음 점 사이의 벡터
    direction = waypoints[i+1] - waypoints[i]
    distance = np.linalg.norm(direction)
    
    if distance > 0:
        # 단위 벡터(방향) * 속력(60km/h)
        velocities[i] = (direction / distance) * target_speed_ms

# 마지막 점은 이전 점과 동일한 속도로 유지 (급정거 방지)
velocities[-1] = velocities[-2]

# 경로 중심점 계산 (카메라용)
center_pos_np = np.mean(waypoints, axis=0)
center_pos_list = [float(center_pos_np[0]), float(center_pos_np[1]), float(center_pos_np[2])]

print("="*60)
print(f"[설정] 경로 포인트 수: {len(waypoints)}개")
print(f"[속도] {target_speed_kmh} km/h ({target_speed_ms:.2f} m/s) 적용 완료")
print("="*60)

# ==============================================================================
# 2. 기지국(Tx) 및 수신기(Rx) 배치
# ==============================================================================
# 객체 초기화
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()):
        scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()):
        scene.remove(name)

# 안테나 설정
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

# 기지국(Tx) 배치
tx_positions = [
    [-125.663, 56.367, -181.453],
    [323.472, 36.869, -204.315],
    [0.663, 56.367, -181.453]
]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(center_pos_list) 
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료.")

# ------------------------------------------------------------------------------
# [수정] 수신기(Rx) 배치 - 위치(position)와 속도(velocity) 동시 입력
# ------------------------------------------------------------------------------
# Mitsuba 호환성을 위해 (N, 3) -> (3, N) 전치(.T) 및 float32 변환 필수
rx = Receiver(name="rx_car", 
              position=waypoints.astype(np.float32).T,
              velocity=velocities.astype(np.float32).T) # 속도 추가

rx.receive_antenna = scene.rx_array
scene.add(rx)

print(" -> 이동하는 자동차(Rx) 배치 완료 (속도 벡터 포함).")

# ==============================================================================
# 3. Ray Tracing 시뮬레이션
# ==============================================================================
print("[연산] 전파 경로 계산 시작...")

solver = PathSolver()

paths = solver(scene, 
               max_depth=3, 
               samples_per_src=100000, 
               diffuse_reflection=True, 
               diffraction=True)

print("[완료] 연산 종료.")

# ==============================================================================
# 4. 시각화 (타임라인 활성화)
# ==============================================================================
print("\n[시각화] 뷰어 실행")
print("1. 뷰어 하단에 **Timeline(슬라이더)**가 생성되었는지 확인하세요.")
print("2. 슬라이더를 움직이면 자동차가 60km/h 속도에 맞춰 이동합니다.")
print("3. 마우스 오른쪽 버튼으로 시점을 조절하여 Top-Down 뷰를 만드세요.")

try:
    # paths만 전달하여 기본 뷰어 실행 (가장 안전한 방법)
    scene.preview(paths=paths, resolution=[800, 600])
except TypeError:
    print(" -> 기본 설정으로 뷰어를 실행합니다.")
    scene.preview(paths=paths)

[설정] 경로 포인트 수: 60개
[속도] 60.0 km/h (16.67 m/s) 적용 완료
 -> Tx_1 배치 완료.
 -> Tx_2 배치 완료.
 -> Tx_3 배치 완료.
 -> 이동하는 자동차(Rx) 배치 완료 (속도 벡터 포함).
[연산] 전파 경로 계산 시작...
[완료] 연산 종료.

[시각화] 뷰어 실행
1. 뷰어 하단에 **Timeline(슬라이더)**가 생성되었는지 확인하세요.
2. 슬라이더를 움직이면 자동차가 60km/h 속도에 맞춰 이동합니다.
3. 마우스 오른쪽 버튼으로 시점을 조절하여 Top-Down 뷰를 만드세요.


일단 기지국이랑 차량이 움직이는건 보임 근데 기지국에서 전파가 나가고 이런 모습은 보이지 않음

전파 보이게끔 수정

In [ ]:
import numpy as np
import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver

# ==============================================================================
# 0. 데이터 준비 & 이동 경로 계산
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

path_indices = [ 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5 

# 거리 및 시간 계산 (60km/h)
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]
speed_ms = 60.0 / 3.6
total_time = total_distance / speed_ms
delta_t = 0.5 

def get_pos_at_time(t):
    target_dist = np.clip(speed_ms * t, 0, total_distance)
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    
    seg_start = cumulative_dists[idx]
    seg_len = segment_dists[idx]
    if seg_len == 0: return waypoints[idx]
    
    ratio = (target_dist - seg_start) / seg_len
    pos = waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])
    return pos  # 이미 numpy array입니다.

time_steps = np.arange(0, total_time + delta_t, delta_t)

# ==============================================================================
# 1. 씬(Scene) 초기화
# ==============================================================================
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()): scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()): scene.remove(name)

scene.tx_array = PlanarArray(num_rows=8, num_cols=8, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[-125.663, 56.367, -181.453], [-50.472, 36.869, -181.453], [0, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at([0,0,0]) 
    scene.add(tx)

start_pos = get_pos_at_time(0.0)
rx = Receiver(name="rx_car", position=start_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)

solver = PathSolver()

# ==============================================================================
# 2. 인터랙티브 위젯 (수정완료)
# ==============================================================================
output_widget = widgets.Output()

def update_simulation(frame_idx):
    t = time_steps[frame_idx]
    current_pos = get_pos_at_time(t) # 여기서 이미 numpy array로 받아옵니다.
    
    # 위치 업데이트
    rx.position = current_pos
    for name in tx_names:
        scene.transmitters[name].look_at(current_pos)
    
    # 경로 계산
    paths = solver(scene, max_depth=3, samples_per_src=100000, 
                   diffuse_reflection=True, diffraction=True)
    
    with output_widget:
        output_widget.clear_output(wait=True)
        
        # 3D 뷰어
        scene.preview(paths=paths, show_devices=True, resolution=[800, 600])
        
        # 정보 출력
        a, _ = paths.cir()
        p_val = tf.reduce_sum(tf.abs(a)**2) if tf.size(a) > 0 else 0.0
        db_val = 10 * np.log10(p_val) if p_val > 0 else -np.inf
        
        # [수정] current_pos.numpy() -> current_pos (이미 numpy 배열이라 메서드 호출 불필요)
        print(f"⏱ Time: {t:.1f}s | 📍 Pos: {current_pos} | 📶 Power: {db_val:.2f} dB")

slider = widgets.IntSlider(
    value=0, min=0, max=len(time_steps)-1, step=1,
    description='Time Step:', layout=widgets.Layout(width='600px')
)

widgets.interactive_output(update_simulation, {'frame_idx': slider})

print("▼ 슬라이더를 움직여보세요.")
display(slider, output_widget)

▼ 슬라이더를 움직여보세요.


IntSlider(value=0, description='Time Step:', layout=Layout(width='600px'), max=101)

Output()